# Retention Copilot: resultados y supuestos

Este notebook explica el reporte agregado generado por `python -m src.train_pipeline`.
No entrena ni optimiza decisiones sobre el test. El beneficio es simulado y depende de los supuestos; no es un efecto causal medido.

La metodología usa 60% entrenamiento, 20% validación y 20% test. Cada modelo se calibra por CV dentro de entrenamiento y se selecciona por beneficio en validación.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd() if (Path.cwd() / 'reports').exists() else Path.cwd().parent
report = json.loads((ROOT / 'reports/executive_report.json').read_text())
assert report['artifact_version'] == 2
print('Modelo:', report['model'])
print('Particiones:', report['split_sizes'])
print('Supuestos:', report['economic_config'])
print('Dataset SHA-256:', report['dataset_sha256'])

## Política de contacto

`P(abandono) × ingreso anual estimado × tasa de éxito − costo > 0`

El ingreso anual es una aproximación basada en saldo, productos y tarjeta; no es CLV de vida completa. Se recomienda no contactar si el beneficio esperado no es positivo. El beneficio incremental de no actuar es cero.

## Selección exclusivamente en validación

Estas métricas comparan candidatos antes de abrir el test. Una diferencia económica pequeña entre candidatos no establece superioridad robusta.

In [ ]:
validation = pd.DataFrame(report['validation_results']).sort_values('net_profit', ascending=False)
validation

## Evaluación del ganador en test reservado

No se vuelve a optimizar modelo o umbral con estas etiquetas. La evaluación retrospectiva aplica una tasa de éxito supuesta a los clientes con abandono contactados.

In [ ]:
pd.Series(report['test_metrics'], name='Test')

In [ ]:
scenarios = pd.DataFrame(report['test_scenarios'])
scenarios

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(scenarios['scenario'], scenarios['net_profit_vs_status_quo_usd'], color=['#94a3b8', '#64748b', '#64748b', '#0d9488'])
ax.set_xlabel('Beneficio incremental simulado (USD)')
ax.set_title('Misma población de test y mismos supuestos')
ax.invert_yaxis()
fig.tight_layout()
plt.show()

In [ ]:
b = report['test_breakdown']
print(f"Contactos: {b['contacted']:,} / {report['split_sizes']['test']:,}")
print(f"Beneficio incremental: US${b['net_profit']:,.2f}")
print(f"ROI incremental: {b['roi']:.1%}")
print(f"Mejora frente a llamar a todos: US${report['uplift_vs_call_everyone']:,.2f}")
print(f"Mejora frente a azar: US${report['uplift_vs_random']:,.2f}")

## Interpretación y límites

- El baseline aleatorio usa la esperanza de seleccionar exactamente la misma cantidad de clientes.
- Los montos son ingreso anual preservado menos costo de contacto; no beneficio contable completo.
- La tasa de éxito no se aprende de intervenciones y se asume igual para todos.
- Una partición aleatoria no valida estabilidad temporal ni resultados en otro banco.
- No se presentan intervalos de confianza ni una evaluación completa de equidad por segmentos.
- La app sobre la base completa puede incluir clientes de entrenamiento: no usar sus cifras como evaluación reservada.

El siguiente paso de negocio sería contrastar estos supuestos mediante una campaña controlada. Para reproducir otras condiciones, ejecutar un nuevo entrenamiento con `--retention-cost` y `--retention-success`; comparar protocolos en validación, sin elegir el más favorable mirando repetidamente el test.